# Live slate

Every game on the newest Pinnacle board. The rotation, shot, and event models are fit once, on completed games before that slate date. Each game then gets its own script: that game's spread, total, posted five, and outs. `BOOK` prices the overs. `N_SIMS` is the number of rows per game. The seed is the matchup, so one game does not depend on another.

Prop probabilities, correlations, expected value, and portfolio risk are the four tables below the run. A correlation is two overs in the same game. A portfolio is one dollar on each of that game's five highest overs. Games do not share rows, so the day book adds those game portfolios as independent.


In [ ]:
import json
import re
import sys
import zlib
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "nba_game_simulator").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("open this notebook from the nba_quant_analysis repo")

REPO = repo_root()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "nba_game_simulator" / "src"))

from game_boxscore import (
    Prop,
    assemble_box_score,
    portfolio_risk,
    price_over,
    prop_probability,
    stat_values,
)
from game_enviroment import (
    build_features,
    build_game_table,
    load_gamelogs,
    load_model,
    player_slate,
    print_sim_summary,
    save_model,
    season_of,
    simulate_game,
    train,
)
from game_opportunities import (
    allocate_game_opportunities,
    fit_opportunities,
    live_opportunities,
    training_opportunities,
)
from game_rotations import (
    allocate_rotation,
    fit_rotation,
    invalid_segment_count,
    play_rates,
    script_exposure,
    script_rates,
    segments_from_minutes,
    training_minutes,
)
from lineup import (
    attach_lineup_fill,
    build_lineup_features,
    fill_lineup_gaps,
    merge_lineup_features,
)
from role_context import attach_script, role_for_slate
from shared_events import allocate_events, fit_events, live_events, training_events
from src.live.predict_points import lineups_on, probable_on, unavailable_on
from src.live.prop_markets import attach_roster, book_names, latest_prop_markets
from src.pipeline.silver.rotowire import normalize_team_abbreviation
from src.scrapers.arbitrage import canonical_team

BOOK = "novig"
N_SIMS = 5000
MODEL_PATH = REPO / "nba_game_simulator" / "game_env_model.json"
PINNACLE_DIR = REPO / "data" / "props" / "pinnacle" / "nba"
LINEUP_DIR = REPO / "data" / "props" / "rotowire_nba_lineup"
STAT_LABELS = (
    ("PTS", "pts"),
    ("REB", "reb"),
    ("AST", "ast"),
    ("STL", "stl"),
    ("BLK", "blk"),
    ("PA", "p+a"),
    ("PRA", "p+r+a"),
    ("RA", "r+a"),
)


In [ ]:
def newest(folder: Path, pattern: str) -> Path:
    files = sorted(path for path in folder.glob(pattern) if path.is_file())
    if not files:
        raise FileNotFoundError(f"no {pattern} in {folder}")
    return files[-1]


def snapshot_date(path: Path) -> str:
    match = re.search(r"\d{4}-\d{2}-\d{2}", path.name)
    if match is None:
        raise ValueError(f"no slate date in {path.name}")
    return match.group(0)


def abbr_for(label: str) -> str:
    wanted = canonical_team(label, "nba")
    for abbr in roster["team_abbreviation"].dropna().unique():
        normalized = normalize_team_abbreviation(abbr)
        if canonical_team(normalized, "nba") == wanted:
            return normalized
    return label


log_paths = sorted((REPO / "data" / "silver" / "nba").glob("*/regular_season/player_gamelogs.parquet"))
if not log_paths:
    raise FileNotFoundError("no player gamelogs under data/silver/nba")
roster_path = log_paths[-1]
played = pd.read_parquet(
    roster_path,
    columns=["player_name", "player_id", "team_abbreviation", "game_date"],
)
played["game_date"] = pd.to_datetime(played["game_date"])
roster = (
    played.sort_values(["player_id", "game_date"])
    .groupby("player_id", as_index=False)
    .tail(1)[["player_name", "player_id", "team_abbreviation"]]
)

odds_path = newest(PINNACLE_DIR, "pinnacle_nba_*_team.json")
lineup_path = newest(LINEUP_DIR, "rotowire_nba_*.json")
slate_date = snapshot_date(odds_path)
if snapshot_date(lineup_path) != slate_date:
    raise ValueError(
        f"odds are {slate_date} and lineups are {snapshot_date(lineup_path)}"
    )
board = json.loads(odds_path.read_text())["games"]
posted = lineups_on(slate_date)
outs = unavailable_on(slate_date)
probable = probable_on(slate_date)

games = []
for game in board:
    participants = game.get("participants") or []
    if len(participants) < 2:
        continue
    away_name, home_name = participants[0], participants[1]
    markets = game.get("team_markets") or {}
    total = None
    for block in markets.get("total") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "over" and line.get("points") is not None:
                total = line["points"]
                break
    home_spread = None
    for block in markets.get("spread") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "home" and line.get("points") is not None:
                home_spread = line["points"]
                break
    home = abbr_for(home_name)
    away = abbr_for(away_name)
    games.append(
        {
            "away": away,
            "home": home,
            "home_spread": home_spread,
            "total": total,
            "away_five": " ".join(posted.get(away) or []) or "—",
            "home_five": " ".join(posted.get(home) or []) or "—",
        }
    )
if not games:
    raise ValueError(f"no games on {odds_path.name}")

print(f"Pinnacle: {odds_path.name}")
print(f"Rotowire: {lineup_path.name}")
print(f"roster:   {roster_path.relative_to(REPO)} through {played['game_date'].max().date()}")
display(pd.DataFrame(games))


In [ ]:
import argparse

saved = json.loads(MODEL_PATH.read_text()) if MODEL_PATH.exists() else None
if saved is not None and saved.get("version") == 3:
    model = load_model(MODEL_PATH)
    print(f"loaded {MODEL_PATH.name}  ({model.get('trained_games')} games)")
else:
    if saved is not None:
        print(
            f"{MODEL_PATH.name} is version {saved.get('version')}; "
            "retraining version 3"
        )
    raw = load_gamelogs(
        str(REPO / "data" / "silver" / "nba" / "*" / "regular_season" / "player_gamelogs.parquet")
    )
    env_games = build_game_table(raw)
    lineups, lineup_state = build_lineup_features(
        raw, halflife=12.0, prior_weight=4.0, season_carry=0.6, window=300,
    )
    env_games = merge_lineup_features(env_games, lineups)
    lineup_state = attach_lineup_fill(lineup_state, env_games)
    env_games = fill_lineup_gaps(env_games)
    env_games, state = build_features(
        env_games, halflife=12.0, prior_w=4.0, carry=0.6, window=300, min_league=30,
    )
    state["lineup"] = lineup_state
    args = argparse.Namespace(
        test_seasons=1,
        validation_season="2025-26",
        holdout_season="2026-27",
        halflife=12.0,
        prior_weight=4.0,
        season_carry=0.6,
        league_window=300,
        spread_convention="auto",
        plots=False,
    )
    model = train(env_games, state, args)
    save_model(model, str(MODEL_PATH))


In [ ]:
ROLE_COLUMNS = [
    "season_year",
    "player_id",
    "player_name",
    "team_id",
    "team_abbreviation",
    "game_id",
    "game_date",
    "matchup",
    "min",
    "pf",
    "team_pace",
    "team_off_rating",
    "team_def_rating",
    "pace",
    "off_rating",
    "def_rating",
    "start_position",
    "team_pts",
    "opp_pts",
    "player_team_spread",
    "fga",
    "fgm",
    "fg3_a",
    "fg3_m",
    "fta",
    "ftm",
    "tov",
    "team_fga",
    "team_poss",
    "team_tov",
    "ast",
    "oreb",
    "dreb",
    "stl",
    "blk",
    "team_fgm",
    "team_ast",
    "team_oreb",
    "team_stl",
    "team_blk",
    "opp_dreb",
    "opp_tov",
    "opp_fga",
]
logs = pd.concat(
    [pd.read_parquet(path, columns=ROLE_COLUMNS) for path in log_paths],
    ignore_index=True,
)
logs["game_date"] = pd.to_datetime(logs["game_date"])
slate_day = pd.Timestamp(slate_date)
history = logs.loc[logs["game_date"] < slate_day]
if history.empty:
    raise ValueError(f"no completed games before {slate_date}")
this_season = history.loc[history["season_year"].astype(str).eq(season_of(slate_day))]
latest = (
    this_season.sort_values(["player_id", "game_date", "game_id"])
    .groupby("player_id", as_index=False)
    .tail(1)
)
latest["abbr"] = latest["team_abbreviation"].map(normalize_team_abbreviation)

trained = training_minutes(history)
rotation = fit_rotation(trained)
shot_rows = training_opportunities(history)
shot_model = fit_opportunities(history)
event_rows = training_events(history)
event_model = fit_events(history)
pbp_all = pd.read_parquet(REPO / "data" / "PlayByPlay" / "s27_pbp.parquet")
print(
    f"rotation fit on {len(trained):,} player-games"
    f" through {history['game_date'].max().date()}"
)
print(
    f"shot model fit on {shot_model.metadata['player_games']:,} player-games"
    f" through {shot_model.training_cutoff}"
    f"  ({len(shot_rows):,} rolling rows)"
)
print(
    f"event model fit through {event_model.cutoff}"
    f"  ({len(event_rows):,} rolling rows)"
)

markets, market_audit = latest_prop_markets(REPO / "data" / "props", BOOK)
others = ", ".join(name for name in book_names() if name != market_audit["book"])
print(
    f"{market_audit['book']} from {market_audit['source_file']}. "
    f"Change BOOK to {others}."
)
if market_audit["book"] == "prizepicks":
    print("PrizePicks has no two-way price. Both sides are -119.")


In [ ]:
def team_id_for(abbr: str) -> int:
    ids = latest.loc[latest["abbr"].eq(abbr), "team_id"].dropna()
    if ids.empty:
        raise ValueError(f"no completed games for {abbr} before {slate_date}")
    return int(ids.mode().iloc[0])


def players_for(abbr, own_id, other_id, is_home, spread, starter_ids, out_ids, probable_ids):
    starters = {int(player_id) for player_id in (starter_ids or [])}
    out_ids = {int(player_id) for player_id in (out_ids or [])}
    probable_ids = {int(player_id) for player_id in (probable_ids or [])}
    side = latest.loc[latest["abbr"].eq(abbr)]
    posted = latest.loc[latest["player_id"].astype(int).isin(starters | out_ids | probable_ids)]
    missing = (starters | out_ids | probable_ids).difference(posted["player_id"].astype(int))
    if missing:
        print(f"{abbr} posted players missing from logs: {sorted(missing)}")
    side = pd.concat([side, posted], ignore_index=True).drop_duplicates("player_id")
    if side.empty:
        raise ValueError(f"no players for {abbr} before {slate_date}")
    on_roster = side["player_id"].astype(int)
    return pd.DataFrame(
        {
            "player_id": side["player_id"].to_numpy(),
            "player_name": side["player_name"].to_numpy(),
            "team": abbr,
            "game_date": slate_day,
            "team_id": own_id,
            "opp_team_id": other_id,
            "is_home": float(bool(is_home)),
            "is_starter": on_roster.isin(starters).astype(float).to_numpy(),
            "probable": on_roster.isin(probable_ids).astype(float).to_numpy(),
            "available": (~on_roster.isin(out_ids)).astype(float).to_numpy(),
            "team_spread": float(spread),
        }
    )


def check_minutes(label, environment, side, drawn, segments):
    row = environment.iloc[0]
    invalid = invalid_segment_count(
        segments[0],
        side,
        game_minutes=float(row["game_minutes"]),
        n_ot=float(row["n_ot"]),
    )
    if invalid != 0:
        raise AssertionError(f"{label} script 0 has {invalid} invalid segments")
    budget = 5.0 * environment["game_minutes"].to_numpy()
    gap = np.max(np.abs(drawn.sum(axis=1) - budget))
    if gap > 1e-4:
        raise AssertionError(f"{label} minutes miss the script budget by {gap:.4f}")
    if np.any(drawn > environment["game_minutes"].to_numpy()[:, None] + 1e-6):
        raise AssertionError(f"{label} has a player above the game length")


def ordered_profile(profile, side, label):
    ordered = profile.set_index("player_id").loc[side["player_id"]].reset_index()
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError(f"{label} profile does not match the rotation")
    return ordered


def run_game(slate):
    team_abbr = normalize_team_abbreviation(slate["team"])
    opp_abbr = normalize_team_abbreviation(slate["opponent"])
    team_id = team_id_for(team_abbr)
    opp_id = team_id_for(opp_abbr)
    players = pd.concat(
        [
            players_for(
                team_abbr,
                team_id,
                opp_id,
                slate["is_home"],
                slate["spread"],
                slate["team_lineup"],
                slate["team_out"],
                slate["team_probable"],
            ),
            players_for(
                opp_abbr,
                opp_id,
                team_id,
                not slate["is_home"],
                -float(slate["spread"]),
                slate["opponent_lineup"],
                slate["opponent_out"],
                slate["opponent_probable"],
            ),
        ],
        ignore_index=True,
    )
    names = players[["player_id", "player_name", "team", "available"]]
    role = role_for_slate(logs, players.drop(columns=["player_name", "team"]))
    role = names.merge(role, on="player_id", how="left", validate="one_to_one")
    role = role.merge(
        players[["player_id", "probable"]],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
    role["side"] = role["team"].eq(team_abbr).map({True: 0, False: 1})
    role = role.sort_values(
        ["side", "is_starter", "min_mean_10"],
        ascending=[True, False, False],
    ).drop(columns="side")
    rates = play_rates(history, role)
    role["play_rate"] = rates["play_rate"].to_numpy()
    recent_ids = []
    for side_id in role["team_id"].drop_duplicates():
        side_games = (
            history.loc[history["team_id"].eq(side_id), ["game_id", "game_date"]]
            .drop_duplicates("game_id")
            .sort_values("game_date")
            .tail(12)["game_id"]
        )
        recent_ids.extend(side_games.astype(str))
    recent_ids = set(recent_ids)
    exposure_logs = history.loc[
        history["game_id"].astype(str).isin(recent_ids),
        ["game_id", "player_id", "team_id", "game_date", "min", "player_name", "start_position"],
    ]
    pbp = pbp_all.loc[pbp_all["gameId"].astype(str).isin(recent_ids)]
    script = script_rates(script_exposure(pbp, exposure_logs), role)
    role["close_rate"] = script["close_rate"].to_numpy()
    role["blowout_rate"] = script["blowout_rate"].to_numpy()
    team_role = role.loc[role["team"].eq(team_abbr)].reset_index(drop=True)
    opp_role = role.loc[role["team"].eq(opp_abbr)].reset_index(drop=True)

    seed = zlib.crc32(f"{slate_date}|{opp_abbr}|{team_abbr}".encode())
    sim = simulate_game(
        model,
        slate["team"],
        slate["opponent"],
        slate["date"],
        total=slate["total"],
        spread=slate["spread"],
        is_home=bool(slate["is_home"]),
        team_lineup=slate["team_lineup"],
        opponent_lineup=slate["opponent_lineup"],
        n=N_SIMS,
        seed=int(seed),
    )
    print_sim_summary(sim, slate["total"], slate["spread"])
    opp_env = sim.copy()
    opp_env["team_margin"] = -opp_env["team_margin"]
    opp_env["is_home"] = 1.0 - opp_env["is_home"].astype(float)
    game_key = f"{slate_date}-{team_abbr}-{opp_abbr}"
    for frame in (sim, opp_env):
        frame["game_id"] = game_key
        frame["game_date"] = slate_day
    rng = np.random.default_rng(int(seed))
    minutes = {
        team_abbr: allocate_rotation(rotation, team_role, sim, rng, method="segments"),
        opp_abbr: allocate_rotation(rotation, opp_role, opp_env, rng, method="segments"),
    }
    schedules = {
        team_abbr: segments_from_minutes(team_role, sim, minutes[team_abbr]),
        opp_abbr: segments_from_minutes(opp_role, opp_env, minutes[opp_abbr]),
    }
    check_minutes(team_abbr, sim, team_role, minutes[team_abbr], schedules[team_abbr])
    check_minutes(opp_abbr, opp_env, opp_role, minutes[opp_abbr], schedules[opp_abbr])

    shot_profile = live_opportunities(history, players.drop(columns=["player_name", "team"]))
    shot_profile["player_id"] = shot_profile["player_id"].astype(players["player_id"].dtype)
    shot_profile = names.drop(columns="available").merge(
        shot_profile, on="player_id", how="left", validate="one_to_one"
    )
    if sim["is_home"].nunique() != 1:
        raise AssertionError("the slate does not have one home flag")
    slate_is_home = bool(sim["is_home"].iloc[0])
    home_abbr, away_abbr = (team_abbr, opp_abbr) if slate_is_home else (opp_abbr, team_abbr)
    home_role, away_role = (team_role, opp_role) if slate_is_home else (opp_role, team_role)
    game = f"{away_abbr} @ {home_abbr}"
    allocated = allocate_game_opportunities(
        shot_model,
        ordered_profile(shot_profile, home_role, "shot"),
        ordered_profile(shot_profile, away_role, "shot"),
        minutes[home_abbr],
        minutes[away_abbr],
        sim,
        rng,
        home_segments=schedules[home_abbr],
        away_segments=schedules[away_abbr],
    )
    shots = {home_abbr: allocated["home"], away_abbr: allocated["away"]}
    for drawn in shots.values():
        if not drawn["metadata"]["segments_used"]:
            raise AssertionError("shots were not kept inside the Layer 3 segments")
    for label, scenario in ((team_abbr, sim["team_pts"]), (opp_abbr, sim["opp_pts"])):
        target = np.floor(scenario.to_numpy(dtype=float) + 0.5).astype(np.int64)
        if not np.array_equal(shots[label]["PTS"].sum(axis=1), target):
            raise AssertionError(f"{label} points miss the scenario score")
    for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
        unavailable = side["available"].to_numpy() == 0
        if unavailable.any() and shots[label]["FGA"][:, unavailable].sum() != 0:
            raise AssertionError(f"{label} gave a shot to an unavailable player")

    event_profile = live_events(history, players.drop(columns=["player_name", "team"]))
    event_profile["player_id"] = event_profile["player_id"].astype(players["player_id"].dtype)
    event_profile = names.drop(columns="available").merge(
        event_profile, on="player_id", how="left", validate="one_to_one"
    )
    availability = {
        team_abbr: team_role["available"].to_numpy() == 1,
        opp_abbr: opp_role["available"].to_numpy() == 1,
    }
    events, event_details = allocate_events(
        event_model,
        {
            team_abbr: ordered_profile(event_profile, team_role, "event"),
            opp_abbr: ordered_profile(event_profile, opp_role, "event"),
        },
        shots,
        minutes,
        rng,
        segments=schedules,
        availability=availability,
        return_details=True,
    )
    if not event_details["segments_used"]:
        raise AssertionError("events were not kept inside the Layer 3 segments")
    for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
        if event_details["sources"][label]["AST"] != "layer4":
            raise AssertionError(f"{label} assists were drawn again")
        if event_details["sources"][label]["OREB"] != "layer4":
            raise AssertionError(f"{label} offensive rebounds were drawn again")
        if not np.array_equal(events[label]["AST"], shots[label]["AST"]):
            raise AssertionError(f"{label} assists are not the Layer 4 assists")
        if not np.array_equal(events[label]["REB"], events[label]["OREB"] + events[label]["DREB"]):
            raise AssertionError(f"{label} rebounds are not offensive plus defensive")
        unavailable = side["available"].to_numpy() == 0
        if unavailable.any():
            for stat in ("AST", "OREB", "DREB", "STL", "BLK"):
                if events[label][stat][:, unavailable].sum() != 0:
                    raise AssertionError(f"{label} gave {stat} to an unavailable player")
    for label, other in ((team_abbr, opp_abbr), (opp_abbr, team_abbr)):
        misses = shots[label]["FGA"].sum(axis=1) - shots[label]["FGM"].sum(axis=1)
        rebounds = events[label]["OREB"].sum(axis=1) + events[other]["DREB"].sum(axis=1)
        if np.any(rebounds > misses):
            raise AssertionError(f"rebounds on {label} misses exceed those misses")
        if np.any(events[other]["STL"].sum(axis=1) > shots[label]["TOV"].sum(axis=1)):
            raise AssertionError(f"steals exceed {label} turnovers")
        if np.any(events[other]["BLK"].sum(axis=1) > misses):
            raise AssertionError(f"blocks on {label} shots exceed its misses")

    box = assemble_box_score(
        sim,
        {"home": home_role["player_id"], "away": away_role["player_id"]},
        {"home": minutes[home_abbr], "away": minutes[away_abbr]},
        {"home": shots[home_abbr], "away": shots[away_abbr]},
        {"home": events[home_abbr], "away": events[away_abbr]},
        check_score=False,
    )
    if box.n_sims != len(sim):
        raise AssertionError("the box score does not have one row per script")

    game_roster = pd.concat(
        [
            home_role.assign(team=home_abbr, side="home"),
            away_role.assign(team=away_abbr, side="away"),
        ],
        ignore_index=True,
    )
    offered = attach_roster(
        markets,
        game_roster.loc[:, ["team", "side", "player_id", "player_name"]],
    )
    slate_dates = sorted(set(pd.to_datetime(offered["slate_date"]).dt.date)) if not offered.empty else []
    if offered.empty:
        print(f"{game}: no {market_audit['book']} lines matched")
    elif slate_dates != [slate_day.date()]:
        print(f"{game}: market dates {slate_dates}; the slate date is {slate_day.date()}")

    ranked_rows = []
    for stat, label in STAT_LABELS:
        for offer in offered.loc[offered["stat"].eq(stat)].itertuples(index=False):
            index = list(box.teams[offer.side].player_ids).index(offer.player_id)
            column = stat_values(box, offer.side, stat)[:, index]
            priced = price_over(
                box,
                Prop(offer.side, offer.player_id, stat, float(offer.line)),
                float(offer.over_american),
            )
            ranked_rows.append(
                {
                    "game": game,
                    "team": offer.team,
                    "player_name": offer.player_name,
                    "stat": label,
                    "min": float(box.teams[offer.side].stats["MIN"][:, index].mean()),
                    "mean": float(column.mean()),
                    "line": float(offer.line),
                    "odds": int(offer.over_american),
                    "prop_over": priced.p_over,
                    "over_ev": priced.ev,
                    "side": offer.side,
                    "player_id": offer.player_id,
                    "box_stat": stat,
                }
            )
    ranked = pd.DataFrame(ranked_rows)
    if ranked.empty:
        print(f"{game}: no priced props")
        return {"priced": ranked, "pairs": ranked, "portfolio": None}

    positive = ranked.loc[ranked["over_ev"] > 0].reset_index(drop=True)
    pair_rows = []
    for left, right in combinations(range(len(positive)), 2):
        a = positive.iloc[left]
        b = positive.iloc[right]
        priced = prop_probability(
            box,
            [
                Prop(a.side, a.player_id, a.box_stat, float(a.line)),
                Prop(b.side, b.player_id, b.box_stat, float(b.line)),
            ],
        )
        if a.player_id == b.player_id:
            link = "same player"
        elif a.team == b.team:
            link = "teammates"
        else:
            link = "opponents"
        pair_rows.append(
            {
                "game": game,
                "link": link,
                "a": f"{a.player_name} {a.stat} {a.line:g}",
                "b": f"{b.player_name} {b.stat} {b.line:g}",
                "p_a": priced.marginals[0],
                "p_b": priced.marginals[1],
                "joint": priced.joint,
                "product": priced.independent_product,
                "gap": priced.joint - priced.independent_product,
            }
        )
    top = ranked.sort_values("over_ev", ascending=False).head(5)
    legs = [
        Prop(row.side, row.player_id, row.box_stat, float(row.line))
        for row in top.itertuples(index=False)
    ]
    held = portfolio_risk(box, legs, top["odds"].tolist())
    names_book = " | ".join(
        f"{row.player_name} {row.stat} {row.line:g}" for row in top.itertuples(index=False)
    )
    portfolio = {
        "game": game,
        "n": held.n,
        "legs": names_book,
        "expected_profit": held.expected_profit,
        "risk": held.risk,
        "p_loss": held.p_loss,
        "joint": held.joint,
        "product": held.independent_product,
    }
    print(
        f"{game}: {len(ranked)} lines, {len(positive)} with over_ev > 0, "
        f"top over_ev {top['over_ev'].iloc[0]:.3f}"
    )
    visible = ranked.drop(columns=["side", "player_id", "box_stat"])
    return {
        "priced": visible,
        "pairs": pd.DataFrame(pair_rows),
        "portfolio": portfolio,
    }


In [ ]:
priced_frames = []
pair_frames = []
portfolio_rows = []
stopped = []
for listed in games:
    try:
        slate = player_slate(board, posted, roster, listed["home"], outs, probable)
    except (ValueError, AssertionError) as exc:
        print(f"{listed['away']} @ {listed['home']} stopped: {exc}")
        stopped.append({"game": f"{listed['away']} @ {listed['home']}", "error": str(exc)})
        continue
    slate["date"] = slate_date
    try:
        result = run_game(slate)
    except (ValueError, AssertionError) as exc:
        print(f"{listed['away']} @ {listed['home']} stopped: {exc}")
        stopped.append({"game": f"{listed['away']} @ {listed['home']}", "error": str(exc)})
        continue
    if not result["priced"].empty:
        priced_frames.append(result["priced"])
    if not result["pairs"].empty:
        pair_frames.append(result["pairs"])
    if result["portfolio"] is not None:
        portfolio_rows.append(result["portfolio"])

priced = pd.concat(priced_frames, ignore_index=True) if priced_frames else pd.DataFrame()
pairs = pd.concat(pair_frames, ignore_index=True) if pair_frames else pd.DataFrame()
portfolios = pd.DataFrame(portfolio_rows)
print(
    f"{len(games) - len(stopped)} games priced, {len(stopped)} stopped, "
    f"{len(priced)} lines"
)
if stopped:
    display(pd.DataFrame(stopped))


## Prop probabilities

`prop_over` is the share of that game's rows strictly above the line. `mean` is the average of the stat on those rows. A combo is the sum of its pieces on the same row.


In [ ]:
def show(frame, digits):
    shown = frame.copy()
    for column, places in digits.items():
        if column in shown.columns:
            shown[column] = shown[column].round(places)
    display(shown.reset_index(drop=True))


if priced.empty:
    print("no priced props")
else:
    print(f"{len(priced)} overs on {slate_date}, {market_audit['book']}")
    for game, frame in priced.groupby("game", sort=False):
        print(game)
        show(
            frame.drop(columns="game").sort_values("prop_over", ascending=False),
            {"min": 2, "mean": 2, "prop_over": 3, "over_ev": 3},
        )


## Correlations

Each row is two overs with positive expected value in the same game. `joint` is the share of rows where both clear. `product` multiplies the two separate chances. `gap` is joint minus product. The top of a game is the parlay. The bottom is the pair of singles.


In [ ]:
if pairs.empty:
    print("no positive-EV pairs")
else:
    print(f"{len(pairs)} positive-EV pairs on {slate_date}")
    for game, frame in pairs.groupby("game", sort=False):
        ordered = frame.sort_values("gap", ascending=False)
        shown = ordered if len(ordered) <= 16 else pd.concat([ordered.head(8), ordered.tail(8)])
        print(f"{game}  {len(ordered)} pairs, showing {len(shown)}")
        show(
            shown.drop(columns="game"),
            {"p_a": 3, "p_b": 3, "joint": 3, "product": 3, "gap": 3},
        )


## Expected value

`over_ev` is the profit per dollar on that over. A push returns the stake. These are the highest overs on the slate, across games.


In [ ]:
if priced.empty:
    print("no priced props")
else:
    top_day = priced.sort_values("over_ev", ascending=False).head(15)
    print(f"top {len(top_day)} overs by expected value on {slate_date}")
    show(top_day, {"min": 2, "mean": 2, "prop_over": 3, "over_ev": 3})


## Portfolio risk

Each game stakes one dollar on its five highest overs. `expected_profit` and `risk` are the mean and standard deviation of the combined profit on that game's rows. `joint` is the share of rows where all five clear. The day book adds the game portfolios. The games are separate scripts, so the day's risk is the square root of the summed variances.


In [ ]:
if portfolios.empty:
    print("no portfolios")
else:
    print(f"one portfolio per game, $1 on each of its top overs, {market_audit['book']}")
    show(
        portfolios,
        {
            "expected_profit": 3,
            "risk": 3,
            "p_loss": 3,
            "joint": 3,
            "product": 3,
        },
    )
    day_profit = float(portfolios["expected_profit"].sum())
    day_risk = float(np.sqrt((portfolios["risk"] ** 2).sum()))
    print(
        f"day book of these {len(portfolios)} portfolios. "
        f"expected profit = {day_profit:.3f}; risk = {day_risk:.3f}"
    )
